# Faster R-CNN 从零实现：包裹框检测流水线

## 面试问题

面试时我会把 Faster R-CNN 拆成共享 Backbone、RPN 和 RoI Head。RPN 在特征图每个位置对 anchors 预测 objectness 与四个框偏移，训练时用 IoU 分配正负标签。proposal 解码、裁剪和 NMS 后，RoI Pooling 把不同尺寸候选变成固定维度，第二阶段再做前景分类与框回归。训练和评估必须统一坐标约定，NMS 还必须按图像与类别分别执行。下面不用 torchvision 检测器，手写 anchors、IoU、框编码解码、NMS、RPN、RoI pooling 和二阶段 head，并展示真实梯度与逐图检测。

## 真实案例

案例是六张 24×24 仓库顶视灰度图，每张包含一个位置不同的 8×8 亮包裹，并给出人工标注框。字符缩略图展示输入；这里只检测单类包裹，省略多尺度、遮挡和独立测试集。

本实验是用于解释机制的确定性小样本，所有指标均标记为“教学实验”，不能外推为线上收益。

In [1]:
import torch  # 导入 PyTorch 以实现检测流水线与梯度训练。
from torch import nn  # 导入卷积和全连接等基础层。
import torch.nn.functional as F  # 导入池化、激活与损失函数。
torch.manual_seed(47)  # 固定随机种子以复现实验输出。
torch.set_num_threads(1)  # 限制 CPU 线程以稳定小实验耗时。
image_names = ["仓位-左上", "仓位-右上", "仓位-左下", "仓位-右下", "仓位-中上", "仓位-中下"]  # 定义六个可读仓位样本名。
ground_truth_boxes = torch.tensor([[1.0, 1.0, 9.0, 9.0], [13.0, 1.0, 21.0, 9.0], [1.0, 13.0, 9.0, 21.0], [13.0, 13.0, 21.0, 21.0], [7.0, 1.0, 15.0, 9.0], [7.0, 13.0, 15.0, 21.0]], dtype=torch.float32)  # 保存六个人工标注包裹框。
images = torch.full((6, 1, 24, 24), 0.03, dtype=torch.float32)  # 创建六张暗背景仓库图像。
for image_index, box in enumerate(ground_truth_boxes):  # 逐图把亮包裹写入标注框内部。
    x_one, y_one, x_two, y_two = [int(value) for value in box.tolist()]  # 把当前框坐标转成像素整数。
    images[image_index, 0, y_one:y_two, x_one:x_two] = 1.0  # 在对应区域填充包裹亮度。
    images[image_index, 0, y_one + 2:y_two:3, x_one:x_two] = 0.65  # 加入横向纸箱纹理避免纯色空壳。
print("字符缩略图：# 表示包裹，. 表示背景")  # 解释后续缩略图符号。
for image_index, image_name in enumerate(image_names):  # 逐张显示输入图与真实框。
    thumbnail = F.avg_pool2d(images[image_index:image_index + 1], kernel_size=2).squeeze()  # 把二十四像素图降为十二像素字符图。
    print(f"\n{image_name}  GT={ground_truth_boxes[image_index].tolist()}")  # 输出当前仓位名和人工框坐标。
    for row in thumbnail:  # 逐行把缩略像素转成字符。
        print("".join("#" if value > 0.3 else "." for value in row.tolist()))  # 输出当前缩略图行。
print(f"图像批次形状={tuple(images.shape)}，每图目标数=1")  # 汇总检测输入规模。

字符缩略图：# 表示包裹，. 表示背景

仓位-左上  GT=[1.0, 1.0, 9.0, 9.0]
.###........
#####.......
#####.......
#####.......
.###........
............
............
............
............
............
............
............

仓位-右上  GT=[13.0, 1.0, 21.0, 9.0]
.......###..
......#####.
......#####.
......#####.
.......###..
............
............
............
............
............
............
............

仓位-左下  GT=[1.0, 13.0, 9.0, 21.0]
............
............
............
............
............
............
.###........
#####.......
#####.......
#####.......
.###........
............

仓位-右下  GT=[13.0, 13.0, 21.0, 21.0]
............
............
............
............
............
............
.......###..
......#####.
......#####.
......#####.
.......###..
............

仓位-中上  GT=[7.0, 1.0, 15.0, 9.0]
....###.....
...#####....
...#####....
...#####....
....###.....
............
............
............
............
............
............
............

仓位-中下  GT=[7.0, 13.0, 15.0, 21

In [2]:
def box_iou(boxes_one, boxes_two):  # 手写两组 xyxy 框的两两 IoU 矩阵。
    top_left = torch.maximum(boxes_one[:, None, :2], boxes_two[None, :, :2])  # 计算每对交集框左上角。
    bottom_right = torch.minimum(boxes_one[:, None, 2:], boxes_two[None, :, 2:])  # 计算每对交集框右下角。
    intersection_size = (bottom_right - top_left).clamp(min=0.0)  # 截断无交集框的负宽高。
    intersection = intersection_size[:, :, 0] * intersection_size[:, :, 1]  # 计算每对框交集面积。
    area_one = (boxes_one[:, 2] - boxes_one[:, 0]) * (boxes_one[:, 3] - boxes_one[:, 1])  # 计算第一组框面积。
    area_two = (boxes_two[:, 2] - boxes_two[:, 0]) * (boxes_two[:, 3] - boxes_two[:, 1])  # 计算第二组框面积。
    union = area_one[:, None] + area_two[None, :] - intersection  # 根据容斥计算每对框并集面积。
    return intersection / union.clamp(min=1e-6)  # 返回零到一的 IoU 矩阵。
def encode_boxes(anchors, target_boxes):  # 把目标框编码成相对 anchor 的中心与尺度偏移。
    anchor_width = anchors[:, 2] - anchors[:, 0]  # 计算 anchor 宽度。
    anchor_height = anchors[:, 3] - anchors[:, 1]  # 计算 anchor 高度。
    anchor_center_x = (anchors[:, 0] + anchors[:, 2]) * 0.5  # 计算 anchor 横向中心。
    anchor_center_y = (anchors[:, 1] + anchors[:, 3]) * 0.5  # 计算 anchor 纵向中心。
    target_width = target_boxes[:, 2] - target_boxes[:, 0]  # 计算目标框宽度。
    target_height = target_boxes[:, 3] - target_boxes[:, 1]  # 计算目标框高度。
    target_center_x = (target_boxes[:, 0] + target_boxes[:, 2]) * 0.5  # 计算目标框横向中心。
    target_center_y = (target_boxes[:, 1] + target_boxes[:, 3]) * 0.5  # 计算目标框纵向中心。
    delta_x = (target_center_x - anchor_center_x) / anchor_width  # 编码横向中心偏移。
    delta_y = (target_center_y - anchor_center_y) / anchor_height  # 编码纵向中心偏移。
    delta_width = torch.log(target_width / anchor_width)  # 编码宽度比例的对数。
    delta_height = torch.log(target_height / anchor_height)  # 编码高度比例的对数。
    return torch.stack([delta_x, delta_y, delta_width, delta_height], dim=1)  # 合并四个回归目标。
def decode_boxes(anchors, deltas):  # 把 RPN 或 RoI Head 偏移解码回 xyxy 框。
    anchor_width = anchors[:, 2] - anchors[:, 0]  # 计算 anchor 宽度。
    anchor_height = anchors[:, 3] - anchors[:, 1]  # 计算 anchor 高度。
    anchor_center_x = (anchors[:, 0] + anchors[:, 2]) * 0.5  # 计算 anchor 横向中心。
    anchor_center_y = (anchors[:, 1] + anchors[:, 3]) * 0.5  # 计算 anchor 纵向中心。
    center_x = anchor_center_x + deltas[:, 0] * anchor_width  # 解码预测框横向中心。
    center_y = anchor_center_y + deltas[:, 1] * anchor_height  # 解码预测框纵向中心。
    width = anchor_width * torch.exp(deltas[:, 2].clamp(-2.0, 2.0))  # 解码并限制预测框宽度尺度。
    height = anchor_height * torch.exp(deltas[:, 3].clamp(-2.0, 2.0))  # 解码并限制预测框高度尺度。
    return torch.stack([center_x - width * 0.5, center_y - height * 0.5, center_x + width * 0.5, center_y + height * 0.5], dim=1)  # 合并成 xyxy 框。
def clip_boxes(boxes, image_size):  # 把预测框限制在图像边界内。
    clipped = boxes.clone()  # 复制框张量以避免修改原输入。
    clipped[:, 0::2] = clipped[:, 0::2].clamp(0.0, float(image_size))  # 限制所有横坐标范围。
    clipped[:, 1::2] = clipped[:, 1::2].clamp(0.0, float(image_size))  # 限制所有纵坐标范围。
    return clipped  # 返回边界内的预测框。
def nms(boxes, scores, threshold):  # 手写按得分贪心抑制重叠框的 NMS。
    order = torch.argsort(scores, descending=True)  # 按 objectness 从高到低排列候选编号。
    kept = []  # 保存最终保留的候选编号。
    while order.numel() > 0:  # 只要还有未处理候选就继续选择。
        current = int(order[0].item())  # 取当前最高分候选作为保留框。
        kept.append(current)  # 把当前候选编号加入结果。
        if order.numel() == 1:  # 最后只剩一个候选时可以结束。
            break  # 退出循环避免对空张量计算 IoU。
        remaining = order[1:]  # 读取除当前框外的剩余候选。
        overlaps = box_iou(boxes[current:current + 1], boxes[remaining]).squeeze(0)  # 计算当前框与剩余框的 IoU。
        order = remaining[overlaps <= threshold]  # 只保留重叠未超过阈值的候选。
    return torch.tensor(kept, dtype=torch.long)  # 返回按得分顺序保留的候选编号。
feature_centers = torch.arange(12, dtype=torch.float32) * 2.0 + 1.0  # 计算步长二特征图对应的像素中心。
grid_y, grid_x = torch.meshgrid(feature_centers, feature_centers, indexing="ij")  # 创建十二乘十二 anchor 中心网格。
anchors = torch.stack([grid_x.reshape(-1) - 4.0, grid_y.reshape(-1) - 4.0, grid_x.reshape(-1) + 4.0, grid_y.reshape(-1) + 4.0], dim=1)  # 在每个位置生成一个八乘八 anchor。
print(f"anchor 数={len(anchors)}，前五个 anchors={anchors[:5].tolist()}")  # 展示 RPN 实际面对的候选坐标。

anchor 数=144，前五个 anchors=[[-3.0, -3.0, 5.0, 5.0], [-1.0, -3.0, 7.0, 5.0], [1.0, -3.0, 9.0, 5.0], [3.0, -3.0, 11.0, 5.0], [5.0, -3.0, 13.0, 5.0]]


## 基线：所有图都输出固定中心框

固定中心框不读取图像，用最佳框 IoU 与 Recall@0.5 作为和两阶段检测器相同的指标。

In [3]:
fixed_box = torch.tensor([[8.0, 8.0, 16.0, 16.0]], dtype=torch.float32)  # 定义不读取图像的固定中心框。
baseline_ious = torch.cat([box_iou(fixed_box, ground_truth_boxes[index:index + 1]).reshape(1) for index in range(len(images))])  # 逐图计算固定框与 GT 的 IoU。
baseline_recall = (baseline_ious >= 0.5).float().mean().item()  # 计算固定框的 Recall@0.5。
print("图像      固定框 IoU  是否召回")  # 打印逐图基线表头。
for index, image_name in enumerate(image_names):  # 遍历六张图观察固定框表现。
    print(f"{image_name:<7}  {baseline_ious[index]:.3f}       {int(baseline_ious[index] >= 0.5)}")  # 输出当前图的 IoU 与召回标记。
print(f"固定中心框 Recall@0.5={baseline_recall:.1%}，平均 IoU={baseline_ious.mean().item():.3f}")  # 汇总同口径基线指标。

图像      固定框 IoU  是否召回
仓位-左上    0.008       0
仓位-右上    0.024       0
仓位-左下    0.024       0
仓位-右下    0.076       0
仓位-中上    0.058       0
仓位-中下    0.196       0
固定中心框 Recall@0.5=0.0%，平均 IoU=0.064


## 手写核心：共享 Backbone、RPN 目标分配与 proposal 生成

RPN 对 144 个 anchors 同时输出 objectness 和四维偏移。正样本要求 IoU≥0.5，负样本要求 IoU<0.1，中间区域忽略；框回归只在正 anchors 上训练。

In [4]:
class TinyRPN(nn.Module):  # 定义共享特征与单 anchor RPN 头。
    def __init__(self):  # 创建 Backbone、RPN 卷积和两个输出头。
        super().__init__()  # 初始化父类以注册全部参数。
        self.backbone = nn.Conv2d(1, 8, kernel_size=3, stride=2, padding=1)  # 把二十四像素图降为十二像素共享特征图。
        self.rpn_convolution = nn.Conv2d(8, 8, kernel_size=3, padding=1)  # 聚合每个 anchor 周围的局部视觉证据。
        self.objectness_head = nn.Conv2d(8, 1, kernel_size=1)  # 为每个位置输出一个前景 logit。
        self.box_head = nn.Conv2d(8, 4, kernel_size=1)  # 为每个位置输出四个 anchor 偏移。
    def forward(self, image_batch):  # 计算共享特征、objectness 和框偏移。
        features = torch.relu(self.backbone(image_batch))  # 提取十二乘十二的共享图像特征。
        hidden = torch.relu(self.rpn_convolution(features))  # 计算 RPN 局部隐藏表示。
        objectness = self.objectness_head(hidden).flatten(1)  # 展平为每图一百四十四个前景 logits。
        deltas = self.box_head(hidden).permute(0, 2, 3, 1).reshape(image_batch.shape[0], -1, 4)  # 整理每个 anchor 的四维偏移。
        return features, objectness, deltas  # 返回共享特征和 RPN 两个预测分支。
rpn_labels = []  # 保存每张图对全部 anchors 的正负忽略标签。
rpn_targets = []  # 保存每张图对全部 anchors 的框回归目标。
positive_counts = []  # 保存每张图的正 anchor 数便于检查目标分配。
for image_index in range(len(images)):  # 逐图根据 GT 给 anchors 分配监督信号。
    overlaps = box_iou(anchors, ground_truth_boxes[image_index:image_index + 1]).squeeze(1)  # 计算一百四十四个 anchors 对当前 GT 的 IoU。
    labels_for_image = torch.full((len(anchors),), -1.0)  # 用负一初始化需要忽略的中间 IoU anchors。
    labels_for_image[overlaps < 0.1] = 0.0  # 把低重叠 anchors 标为背景负例。
    labels_for_image[overlaps >= 0.5] = 1.0  # 把高重叠 anchors 标为前景正例。
    repeated_target = ground_truth_boxes[image_index:image_index + 1].repeat(len(anchors), 1)  # 为每个 anchor 复制同一个当前 GT。
    targets_for_image = encode_boxes(anchors, repeated_target)  # 编码全部 anchor 到 GT 的回归目标。
    rpn_labels.append(labels_for_image)  # 保存当前图的 anchor 分类标签。
    rpn_targets.append(targets_for_image)  # 保存当前图的 anchor 回归目标。
    positive_counts.append(int((labels_for_image == 1.0).sum().item()))  # 记录当前图正 anchors 数。
rpn_labels = torch.stack(rpn_labels)  # 合并六张图的 RPN 分类标签。
rpn_targets = torch.stack(rpn_targets)  # 合并六张图的 RPN 回归目标。
rpn = TinyRPN()  # 实例化手写 RPN 与共享 Backbone。
print(rpn)  # 展示 RPN 实际层次结构。
print(f"每图正 anchor 数={positive_counts}，标签张量={tuple(rpn_labels.shape)}")  # 展示 IoU 目标分配的中间结果。

TinyRPN(
  (backbone): Conv2d(1, 8, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (rpn_convolution): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (objectness_head): Conv2d(8, 1, kernel_size=(1, 1), stride=(1, 1))
  (box_head): Conv2d(8, 4, kernel_size=(1, 1), stride=(1, 1))
)
每图正 anchor 数=[5, 5, 5, 5, 5, 5]，标签张量=(6, 144)


In [5]:
rpn_optimizer = torch.optim.Adam(rpn.parameters(), lr=0.02)  # 创建优化器学习 objectness 与框偏移。
rpn_loss_trace = []  # 保存 RPN 总损失轨迹。
first_rpn_gradient = 0.0  # 预留首轮 Backbone 梯度范数。
for epoch in range(301):  # 在六张教学图上执行三百零一次 RPN 更新。
    rpn_optimizer.zero_grad()  # 清空上一轮累计梯度。
    feature_maps, objectness_logits, predicted_deltas = rpn(images)  # 运行共享 Backbone 与 RPN 前向传播。
    valid_mask = rpn_labels >= 0.0  # 找出参与 objectness 损失的正负 anchors。
    positive_mask = rpn_labels == 1.0  # 找出参与框回归损失的正 anchors。
    objectness_loss = F.binary_cross_entropy_with_logits(objectness_logits[valid_mask], rpn_labels[valid_mask])  # 计算有效 anchors 的前景二元交叉熵。
    regression_loss = F.smooth_l1_loss(predicted_deltas[positive_mask], rpn_targets[positive_mask])  # 计算正 anchors 的框偏移损失。
    rpn_loss = objectness_loss + regression_loss  # 合并 RPN 分类和回归损失。
    rpn_loss.backward()  # 反向传播到 RPN 与共享 Backbone。
    if epoch == 0:  # 首轮记录真实 Backbone 梯度规模。
        first_rpn_gradient = rpn.backbone.weight.grad.norm().item()  # 读取共享卷积核的首轮梯度范数。
    rpn_optimizer.step()  # 根据当前梯度更新 RPN 参数。
    rpn_loss_trace.append(rpn_loss.item())  # 保存当前轮总损失。
    if epoch in [0, 50, 150, 300]:  # 选择关键轮次输出 RPN 学习轨迹。
        print(f"epoch={epoch:03d} total={rpn_loss.item():.4f} objectness={objectness_loss.item():.4f} box={regression_loss.item():.4f}")  # 输出 RPN 两项损失的真实变化。
rpn.eval()  # 切换到评估模式生成 proposals。
with torch.no_grad():  # 关闭 proposal 阶段的梯度记录。
    feature_maps, objectness_logits, predicted_deltas = rpn(images)  # 重新计算共享特征和 RPN 输出。
proposal_sets = []  # 保存每张图经过 top-k 与 NMS 的 proposals。
proposal_score_sets = []  # 保存每张图对应的 proposal objectness。
for image_index in range(len(images)):  # 逐图解码、裁剪并抑制 proposals。
    decoded = clip_boxes(decode_boxes(anchors, predicted_deltas[image_index]), 24)  # 把当前图的 anchor 偏移解码为像素框。
    scores = torch.sigmoid(objectness_logits[image_index])  # 把前景 logits 转成 objectness 概率。
    top_indices = torch.topk(scores, k=30).indices  # 只保留得分最高的三十个候选供 NMS。
    top_boxes = decoded[top_indices]  # 读取 top-k 候选框。
    top_scores = scores[top_indices]  # 读取 top-k 候选分数。
    kept_indices = nms(top_boxes, top_scores, threshold=0.4)[:8]  # 手写 NMS 并最多保留八个 proposals。
    proposal_sets.append(top_boxes[kept_indices])  # 保存当前图最终 proposals。
    proposal_score_sets.append(top_scores[kept_indices])  # 保存当前图最终 proposal 分数。
print(f"首轮 Backbone 梯度范数={first_rpn_gradient:.6f}")  # 输出非零梯度证明 RPN 被真实训练。
print(f"仓位-左上 top proposals={[(box.tolist(), round(score.item(), 3)) for box, score in zip(proposal_sets[0][:3], proposal_score_sets[0][:3])]}")  # 展示一张图的真实候选框与分数。

epoch=000 total=0.7919 objectness=0.7662 box=0.0257
epoch=050 total=0.0426 objectness=0.0394 box=0.0032


epoch=150 total=0.0284 objectness=0.0257 box=0.0027


epoch=300 total=0.0173 objectness=0.0151 box=0.0021
首轮 Backbone 梯度范数=0.014676
仓位-左上 top proposals=[([0.7007832527160645, 2.9183311462402344, 8.748054504394531, 11.020909309387207], 0.713), ([0.7007832527160645, 0.0, 8.748054504394531, 7.020909309387207], 0.713), ([0.0, 10.640295028686523, 0.0, 15.372344970703125], 0.123)]


## 二阶段：手写 RoI Pooling 与分类回归头

训练 RoI 使用每个 GT 周围的抖动正框和远离 GT 的背景框。RoI Pooling 将像素坐标映射到共享特征图，再把不同大小裁剪池化成 2×2。

In [6]:
def roi_pool(feature_batch, boxes, image_indices, output_size=2):  # 手写逐 RoI 裁剪并池化为固定大小。
    pooled_regions = []  # 保存每个候选框对应的固定形状特征。
    for roi_index, box in enumerate(boxes):  # 逐候选执行坐标映射和裁剪。
        scaled = box / 2.0  # 根据 Backbone 步长二把像素框映射到特征坐标。
        x_one = max(0, min(11, int(torch.floor(scaled[0]).item())))  # 计算并裁剪特征框左边界。
        y_one = max(0, min(11, int(torch.floor(scaled[1]).item())))  # 计算并裁剪特征框上边界。
        x_two = max(x_one + 1, min(12, int(torch.ceil(scaled[2]).item())))  # 计算并保证特征框右边界有效。
        y_two = max(y_one + 1, min(12, int(torch.ceil(scaled[3]).item())))  # 计算并保证特征框下边界有效。
        crop = feature_batch[image_indices[roi_index]:image_indices[roi_index] + 1, :, y_one:y_two, x_one:x_two]  # 从对应图像共享特征中裁剪 RoI。
        pooled = F.adaptive_max_pool2d(crop, (output_size, output_size))  # 把任意大小裁剪池化成二乘二。
        pooled_regions.append(pooled.squeeze(0))  # 保存去掉单样本批次维的 RoI 特征。
    return torch.stack(pooled_regions)  # 合并所有 RoI 的固定形状特征。
class RoIHead(nn.Module):  # 定义 Faster R-CNN 第二阶段分类与框回归头。
    def __init__(self):  # 创建共享隐藏层、前景分类头和框偏移头。
        super().__init__()  # 初始化父类以注册全部参数。
        self.hidden_layer = nn.Linear(8 * 2 * 2, 24)  # 把二乘二共享特征映射成 RoI 隐藏表示。
        self.classifier = nn.Linear(24, 2)  # 输出背景和包裹两个类别 logits。
        self.box_regressor = nn.Linear(24, 4)  # 输出候选框到 GT 的四维精修偏移。
    def forward(self, pooled_features):  # 对固定形状 RoI 特征执行二阶段预测。
        flattened = pooled_features.flatten(1)  # 展平每个 RoI 的通道与空间维。
        hidden = torch.relu(self.hidden_layer(flattened))  # 提取候选区域的非线性表示。
        return self.classifier(hidden), self.box_regressor(hidden), hidden  # 返回类别 logits、框偏移和隐藏表示。
corner_candidates = torch.tensor([[0.0, 0.0, 6.0, 6.0], [18.0, 0.0, 24.0, 6.0], [0.0, 18.0, 6.0, 24.0], [18.0, 18.0, 24.0, 24.0]], dtype=torch.float32)  # 定义四个候选背景角落。
training_rois = []  # 保存二阶段训练 RoIs。
training_roi_images = []  # 保存每个训练 RoI 所属图像编号。
training_roi_labels = []  # 保存每个训练 RoI 的前景背景标签。
training_roi_targets = []  # 保存每个训练 RoI 的框回归目标。
for image_index, ground_truth in enumerate(ground_truth_boxes):  # 逐图构造一个抖动正 RoI 和一个远端背景 RoI。
    positive_roi = clip_boxes((ground_truth + torch.tensor([-1.0, -1.0, 1.0, 1.0])).unsqueeze(0), 24).squeeze(0)  # 扩大 GT 得到需要精修的正候选。
    background_overlaps = box_iou(corner_candidates, ground_truth.unsqueeze(0)).squeeze(1)  # 计算四个角落与当前 GT 的重叠。
    background_roi = corner_candidates[torch.argmin(background_overlaps)]  # 选择重叠最小的角落作为背景候选。
    training_rois.extend([positive_roi, background_roi])  # 加入当前图的正负两个 RoI。
    training_roi_images.extend([image_index, image_index])  # 记录两个 RoI 都来自当前图。
    training_roi_labels.extend([1, 0])  # 标记正 RoI 为包裹并标记角落为背景。
    training_roi_targets.extend([encode_boxes(positive_roi.unsqueeze(0), ground_truth.unsqueeze(0)).squeeze(0), torch.zeros(4)])  # 保存正框精修目标和背景占位目标。
training_rois = torch.stack(training_rois)  # 合并十二个训练 RoIs。
training_roi_images = torch.tensor(training_roi_images, dtype=torch.long)  # 转换 RoI 图像编号为张量。
training_roi_labels = torch.tensor(training_roi_labels, dtype=torch.long)  # 转换 RoI 分类标签为张量。
training_roi_targets = torch.stack(training_roi_targets)  # 合并 RoI 框回归目标。
roi_head = RoIHead()  # 实例化手写二阶段 RoI Head。
roi_optimizer = torch.optim.Adam(roi_head.parameters(), lr=0.02)  # 创建优化器训练分类与精修分支。
roi_loss_trace = []  # 保存二阶段总损失轨迹。
detached_features = feature_maps.detach()  # 固定已训练 Backbone 以单独讲清二阶段目标。
training_pooled = roi_pool(detached_features, training_rois, training_roi_images)  # 对十二个训练 RoIs 执行手写池化。
for epoch in range(251):  # 执行二百五十一次二阶段参数更新。
    roi_optimizer.zero_grad()  # 清空上一轮二阶段梯度。
    roi_logits, roi_deltas, roi_hidden = roi_head(training_pooled)  # 对固定 RoI 特征执行分类和回归前向传播。
    classification_loss = F.cross_entropy(roi_logits, training_roi_labels)  # 计算前景背景二分类交叉熵。
    positive_roi_mask = training_roi_labels == 1  # 找出需要框精修的正 RoIs。
    refinement_loss = F.smooth_l1_loss(roi_deltas[positive_roi_mask], training_roi_targets[positive_roi_mask])  # 只在正 RoIs 上计算框回归损失。
    roi_loss = classification_loss + refinement_loss  # 合并二阶段分类与回归损失。
    roi_loss.backward()  # 反向传播到 RoI Head 两个输出分支。
    roi_optimizer.step()  # 根据当前梯度更新二阶段参数。
    roi_loss_trace.append(roi_loss.item())  # 保存当前轮二阶段损失。
    if epoch in [0, 50, 150, 250]:  # 选择关键轮次输出训练轨迹。
        roi_accuracy = (roi_logits.argmax(dim=1) == training_roi_labels).float().mean().item()  # 计算当前 RoI 分类准确率。
        print(f"roi_epoch={epoch:03d} loss={roi_loss.item():.4f} roi_accuracy={roi_accuracy:.1%}")  # 输出二阶段损失与准确率。
roi_head.eval()  # 切换到评估模式执行最终 proposal 筛选。
detection_boxes = []  # 保存每张图二阶段选出的检测框。
detection_scores = []  # 保存每张图最终包裹概率。
detection_ious = []  # 保存每张图检测框与 GT 的 IoU。
with torch.no_grad():  # 关闭检测结果生成阶段的梯度记录。
    for image_index in range(len(images)):  # 逐图把 RPN proposals 送入 RoI Head。
        proposal_boxes = proposal_sets[image_index]  # 读取当前图经 NMS 保留的 proposals。
        proposal_images = torch.full((len(proposal_boxes),), image_index, dtype=torch.long)  # 创建每个 proposal 的图像编号。
        pooled = roi_pool(feature_maps, proposal_boxes, proposal_images)  # 池化当前图全部 proposals 的共享特征。
        class_logits, box_deltas, hidden = roi_head(pooled)  # 计算每个 proposal 的包裹分数与精修偏移。
        object_scores = torch.softmax(class_logits, dim=1)[:, 1]  # 提取二阶段包裹类别概率。
        combined_scores = object_scores * proposal_score_sets[image_index]  # 融合 RPN objectness 与二阶段类别分数。
        best_index = int(torch.argmax(combined_scores).item())  # 选择融合分数最高的候选。
        refined = clip_boxes(decode_boxes(proposal_boxes[best_index:best_index + 1], box_deltas[best_index:best_index + 1]), 24)[0]  # 应用二阶段框偏移得到最终框。
        final_iou = box_iou(refined.unsqueeze(0), ground_truth_boxes[image_index:image_index + 1]).item()  # 计算最终框与当前 GT 的 IoU。
        detection_boxes.append(refined)  # 保存当前图最终检测框。
        detection_scores.append(combined_scores[best_index].item())  # 保存当前图最终融合分数。
        detection_ious.append(final_iou)  # 保存当前图最终 IoU。
detector_recall = sum(iou >= 0.5 for iou in detection_ious) / len(detection_ious)  # 计算完整两阶段检测器的 Recall@0.5。
print(f"RoI 池化张量形状={tuple(training_pooled.shape)}，首个正 RoI 隐藏表示前六维={[round(value, 3) for value in roi_hidden[0, :6].detach().tolist()]}")  # 展示二阶段关键中间张量。

roi_epoch=000 loss=0.6668 roi_accuracy=50.0%
roi_epoch=050 loss=0.0000 roi_accuracy=100.0%


roi_epoch=150 loss=0.0000 roi_accuracy=100.0%
roi_epoch=250 loss=0.0000 roi_accuracy=100.0%
RoI 池化张量形状=(12, 8, 2, 2)，首个正 RoI 隐藏表示前六维=[2.061, 0.0, 1.862, 0.0, 0.005, 1.797]


## 结果解读

每张图都输出最终框、融合分数和 IoU，因此能区分“objectness 很高但框不准”与真正召回。这里只报告单目标 Recall@0.5，不把六张训练图冒充完整 mAP 基准。

In [7]:
print("图像      GT框                    检测框                  分数   IoU   召回")  # 打印逐图两阶段检测结果表头。
for image_index, image_name in enumerate(image_names):  # 遍历六张图展示最终检测结果。
    rounded_box = [round(value, 2) for value in detection_boxes[image_index].tolist()]  # 把当前预测框坐标保留两位小数。
    print(f"{image_name:<7}  {ground_truth_boxes[image_index].tolist()}  {rounded_box}  {detection_scores[image_index]:.3f}  {detection_ious[image_index]:.3f}  {int(detection_ious[image_index] >= 0.5)}")  # 输出当前图完整检测证据。
print(f"同数据 Recall@0.5：固定框={baseline_recall:.1%}，两阶段检测器={detector_recall:.1%}")  # 汇总基线与完整检测流水线的同口径指标。

图像      GT框                    检测框                  分数   IoU   召回
仓位-左上    [1.0, 1.0, 9.0, 9.0]  [1.49, 0.59, 7.94, 6.27]  0.713  0.510  1
仓位-右上    [13.0, 1.0, 21.0, 9.0]  [13.48, 0.59, 19.93, 6.28]  0.713  0.511  1
仓位-左下    [1.0, 13.0, 9.0, 21.0]  [1.5, 13.7, 7.94, 20.17]  0.713  0.651  1
仓位-右下    [13.0, 13.0, 21.0, 21.0]  [14.97, 6.77, 19.69, 15.07]  0.710  0.104  0
仓位-中上    [7.0, 1.0, 15.0, 9.0]  [7.48, 0.59, 13.93, 6.28]  0.713  0.511  1
仓位-中下    [7.0, 13.0, 15.0, 21.0]  [7.51, 13.68, 13.95, 20.17]  0.713  0.653  1
同数据 Recall@0.5：固定框=0.0%，两阶段检测器=83.3%


## 失败案例：跨图像做一次全局 NMS

不同图像中的框即使坐标完全相同也不是重复目标。若把整个 batch 的框混在一起 NMS，会错误删除另一张图的合法检测；正确实现必须按图像分组。

In [8]:
same_coordinate_boxes = torch.tensor([[1.0, 1.0, 9.0, 9.0], [1.0, 1.0, 9.0, 9.0]], dtype=torch.float32)  # 构造分属两张图但坐标相同的合法检测。
same_coordinate_scores = torch.tensor([0.95, 0.90], dtype=torch.float32)  # 为两个合法检测设置不同置信度。
image_ids = torch.tensor([0, 1], dtype=torch.long)  # 明确两个框来自不同图像。
global_kept = nms(same_coordinate_boxes, same_coordinate_scores, threshold=0.5)  # 复现错误的跨图像全局 NMS。
per_image_kept = []  # 保存按图像分别 NMS 后的全局编号。
for image_id in image_ids.unique():  # 逐图像隔离执行 NMS。
    local_indices = torch.where(image_ids == image_id)[0]  # 找出当前图像对应的候选编号。
    local_kept = nms(same_coordinate_boxes[local_indices], same_coordinate_scores[local_indices], threshold=0.5)  # 只在当前图像内部抑制重复框。
    per_image_kept.extend(local_indices[local_kept].tolist())  # 把局部保留编号映射回全局列表。
print(f"错误全局 NMS 保留编号={global_kept.tolist()}，合法框数={len(global_kept)}")  # 展示另一张图的框被误删。
print(f"修复为按图 NMS 后保留编号={per_image_kept}，合法框数={len(per_image_kept)}")  # 展示两个独立检测都被保留。
print("修复结论：NMS 至少按 image_id 和 class_id 分组执行。")  # 给出批量检测的明确工程门禁。

错误全局 NMS 保留编号=[0]，合法框数=1
修复为按图 NMS 后保留编号=[0, 1]，合法框数=2
修复结论：NMS 至少按 image_id 和 class_id 分组执行。


## 生产差距

真实 Faster R-CNN 使用 FPN、多尺度多比例 anchors、采样平衡、RoIAlign、多类别 NMS 和独立验证集。坐标还涉及 resize、padding 和原图回映射。线上应监控 proposal recall、每类 AP、空图误报、延迟与显存。本实验只有单类、单目标并在训练图上评估，只用于验证流水线。

## 最小回归测试

In [9]:
assert len(image_names) >= 5  # 保证案例包含足够多的可读检测样本。
assert rpn_loss_trace[-1] < rpn_loss_trace[0]  # 保证 RPN 真实训练损失下降。
assert roi_loss_trace[-1] < roi_loss_trace[0]  # 保证二阶段分类回归损失下降。
assert first_rpn_gradient > 0.0  # 保证共享 Backbone 获得了非零梯度。
assert detector_recall > baseline_recall  # 保证完整检测流水线超过固定框基线。
assert detector_recall >= 5 / 6  # 保证至少五张图达到 IoU 0.5。
assert len(global_kept) == 1 and len(per_image_kept) == 2  # 保证跨图 NMS 失败与按图修复均可复现。
print("回归测试通过：RPN、RoI Head、检测召回和 NMS 分组均符合预期。")  # 输出集中断言的最终验收结论。

回归测试通过：RPN、RoI Head、检测召回和 NMS 分组均符合预期。
